# STD Data Analysis
Exploring CDC data on STD cases in the U.S. (2000-2023)

### ============================================
### SECTION 1: LOADING RAW DATA
### ============================================

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

chlamydia = pd.read_csv("../data/raw/CHLAMYDIA_RAW.csv", skiprows=7)

print(chlamydia.head())
print(chlamydia.info())

### ============================================
### SECTION 2: DATA EXPLORATION
### ============================================

In [ ]:
print(chlamydia["Year"].unique()) # Year Range
print(chlamydia["Geography"].nunique()) # Number of States in Dataset
print(chlamydia.isnull().sum()) # Any missing data

columns_to_check = ["Year", "Cases", "Rate per 100000"]

for col in columns_to_check:
    temp = chlamydia.copy()
    # Remove all commas from numbers and replace any non-number values with NaN
    convert_to_numeric = pd.to_numeric(temp[col].str.replace(",", ""), errors="coerce")
    
    # non_numeric_rows = only the rows that have a NaN value
    non_numeric_rows = temp[convert_to_numeric.isna()]

    if not non_numeric_rows.empty:
        print(f"Non-numeric values found in column '{col}':")
        print(non_numeric_rows[["Year", "Geography", "Cases", "Rate per 100000"]])
        print("\n")
    else:
        print(f"All values in column '{col}' are numeric.")

### ============================================
### SECTION 3: CLEANING DATA
### ============================================

In [ ]:
cleaned_chlamydia = chlamydia.copy()

# Removing all non numbers from "Year" column
cleaned_chlamydia["Year"] = cleaned_chlamydia["Year"].str.replace(" (COVID-19 Pandemic)", "")

# Convert all non numbers in the "Cases" and "Rate per 100000" columns to NaN
cleaned_chlamydia["Cases"] = pd.to_numeric(cleaned_chlamydia["Cases"].str.replace(",", ""), errors="coerce")
cleaned_chlamydia["Rate per 100000"] = pd.to_numeric(cleaned_chlamydia["Rate per 100000"], errors="coerce")

# Remove rows that have no data (cant be used)
cleaned_chlamydia = cleaned_chlamydia.dropna(subset=["Cases", "Rate per 100000"], how="any")

# Final Check to see if any values are non-numeric
for col in ["Year", "Cases", "Rate per 100000"]:
    try:
        pd.to_numeric(cleaned_chlamydia[col])
        print(f'All values in the "{col}" column are numeric')
    except:
        print(f'There are still non-numeric values in "{col}".')

cleaned_chlamydia.to_csv("../data/cleaned/CHLAMYDIA_CLEANED.csv")


### ============================================
### SECTION 4: VISUALIZATIONS
### ============================================

In [ ]:
# Chart 1: Cases Over Time
yearly_chart = cleaned_chlamydia.groupby("Year")["Cases"].sum()
yearly_chart.index = pd.to_numeric(yearly_chart.index)
fig, ax = plt.subplots(figsize=(10, 6))
yearly_chart.plot(kind="line", ax=ax, title="Cases Over Time")
ax.set_xlabel("Year")
ax.set_ylabel("Cases", rotation=0, labelpad=30)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig("cases_over_time.png")
plt.show()

# Chart 2: Top 10 States (All Time Cases)
top_states = cleaned_chlamydia.groupby("Geography")["Cases"].sum().sort_values(ascending=False).head(10)
top_states = top_states[::-1]
fig, ax = plt.subplots(figsize=(10, 6))
top_states.plot(kind="barh", ax=ax, title="Top 10 States (All Time Cases)")
ax.set_xlabel("Cases")
ax.set_ylabel("States", rotation=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig("top_states.png")
plt.show()

# Chart 3: Bottom 10 States (All Time Cases)
bottom_states = cleaned_chlamydia.groupby("Geography")["Cases"].sum().sort_values(ascending=True).head(10)
bottom_states = bottom_states[::-1]
fig, ax = plt.subplots(figsize=(10, 6))
bottom_states.plot(kind="barh", ax=ax, title="Bottom 10 States (All Time Cases)")
ax.set_xlabel("Cases")
ax.set_ylabel("States", rotation=0)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig("bottom_states.png")
plt.show()

# Chart 4: Top 10 States (All Time Rate per 100000)
top_states_rate = cleaned_chlamydia.groupby("Geography")["Rate per 100000"].sum().sort_values(ascending=False).head(10)
top_states_rate = top_states_rate[::-1]
fig, ax = plt.subplots(figsize=(10, 6))
top_states_rate.plot(kind="barh", ax=ax, title="Top 10 States (All Time Rate per 100,000)")
ax.set_xlabel("Rate Per 100,000")
ax.set_ylabel("States", rotation=0, labelpad=10)
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f}"))
plt.tight_layout()
plt.savefig("top_states_rate.png")
plt.show()


# Chart 5: State w/ Highest Cases Per Year (Rate Per 100000)
highest_per_year = cleaned_chlamydia.groupby(["Year", "Geography"])["Rate per 100000"].sum().reset_index()
max_cases = highest_per_year.loc[highest_per_year.groupby("Year")["Rate per 100000"].idxmax()]


fig, ax = plt.subplots(figsize=(10,6))
bars = ax.barh(max_cases["Year"], max_cases["Rate per 100000"])
ax.set_xlabel("Rate per 100,000")
ax.set_ylabel("Year", rotation=0, labelpad=15)
ax.set_title("State with the Highest Cases Per Year (Rate Per 100,000)")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f"{x:,.0f}"))

# Adding the state name and Rate per 100,000 in the middle of the bar
for year, rate, state in zip(max_cases["Year"], max_cases["Rate per 100000"], max_cases["Geography"]):
    ax.text(rate / 2, year, state, ha="center", va="center", fontsize=9, color="white")
    ax.text(rate / 2 + (rate * 0.23), year, f"{rate:,.0f}", ha="center", va="center", fontsize=9, color="white")

plt.tight_layout()
plt.savefig("highest_per_year.png")
plt.show()
